In [0]:

---Viewing the table
select * from `snxb_cardealer`.`brightmotors`.`snxb_car_sales_csv_bright_learn_project` limit 100;

--- Timestamp Conversion
SELECT 
  saledate AS raw_saledate,

  -- 1. Convert "Dec 16 2014" to Date
  TO_DATE(CONCAT(SUBSTRING(saledate, 5, 6), ' ', SUBSTRING(saledate, 12, 4)), 'MMM dd yyyy') AS sale_date,

  -- 2. Extract Abbreviated Day of Week directly ("Tue", "Thu")
  SUBSTRING(saledate, 1, 3) AS day_of_week,

  -- 3. Extract Time directly ("12:30:00")
  SUBSTRING(saledate, 17, 8) AS sale_time

FROM `snxb_cardealer`.`brightmotors`.`snxb_car_sales_csv_bright_learn_project`;

-- Enable legacy time parser policy for the current session
SET spark.sql.legacy.timeParserPolicy = LEGACY;

CREATE OR REPLACE VIEW `snxb_cardealer`.`brightmotors`.`silver_car_sales` AS
SELECT 
  -- Select existing key columns from your dataset
  year,
  make,
  model,
  trim,
  body,
  transmission,
  vin,
  state,
  condition,
  odometer,
  color,
  interior,
  seller,
  mmr,
  sellingprice,

  -- Keep the raw date column for audit/reference
  saledate AS raw_saledate,

  -- 1. True SQL DATE type (YYYY-MM-DD)
  CAST(TO_TIMESTAMP(saledate, 'EEE MMM dd yyyy HH:mm:ss') AS DATE) AS sale_date,

  -- 2. Full Day of Week Name (e.g., Tuesday, Thursday)
  CASE SUBSTRING(saledate, 1, 3)
    WHEN 'Mon' THEN 'Monday'
    WHEN 'Tue' THEN 'Tuesday'
    WHEN 'Wed' THEN 'Wednesday'
    WHEN 'Thu' THEN 'Thursday'
    WHEN 'Fri' THEN 'Friday'
    WHEN 'Sat' THEN 'Saturday'
    WHEN 'Sun' THEN 'Sunday'
    ELSE 'Unknown'
  END AS day_of_week,

  -- 3. Time Portion (HH:mm:ss)
  SUBSTRING(saledate, 17, 8) AS sale_time

FROM `snxb_cardealer`.`brightmotors`.`snxb_car_sales_csv_bright_learn_project`;



CREATE OR REPLACE VIEW `snxb_cardealer`.`brightmotors`.`silver_car_sales` AS

WITH cleaned_and_transformed AS (
  SELECT 
    year,
    make,
    model,
    trim,
    body,
    transmission,
    vin,
    state,
    condition,
    odometer,
    color,
    interior,
    
    COALESCE(TRIM(seller), 'Unknown Seller') AS seller,
    mmr,
    sellingprice,
    saledate AS raw_saledate,

    -- 1. Parse Date safely using SUBSTRING without strict pattern errors
    TO_DATE(CONCAT(SUBSTRING(saledate, 5, 6), ' ', SUBSTRING(saledate, 12, 4)), 'MMM dd yyyy') AS sale_date,

    -- 2. Extract Full Day Name safely
    CASE SUBSTRING(saledate, 1, 3)
      WHEN 'Mon' THEN 'Monday'
      WHEN 'Tue' THEN 'Tuesday'
      WHEN 'Wed' THEN 'Wednesday'
      WHEN 'Thu' THEN 'Thursday'
      WHEN 'Fri' THEN 'Friday'
      WHEN 'Sat' THEN 'Saturday'
      WHEN 'Sun' THEN 'Sunday'
      ELSE 'Unknown'
    END AS day_of_week,

    -- 3. Extract Time Portion
    SUBSTRING(saledate, 17, 8) AS sale_time,

    -- Helper column for deduplication across records
    ROW_NUMBER() OVER (
      PARTITION BY vin 
      ORDER BY saledate DESC
    ) AS row_num

  FROM `snxb_cardealer`.`brightmotors`.`snxb_car_sales_csv_bright_learn_project`
  WHERE vin IS NOT NULL AND TRIM(vin) != ''
)

SELECT 
  * EXCEPT (row_num)
FROM cleaned_and_transformed
WHERE row_num = 1;

SELECT * FROM snxb_cardealer.brightmotors.silver_car_sales LIMIT 100;


---Sliver Cleaning
CREATE OR REPLACE VIEW `snxb_cardealer`.`brightmotors`.`silver_car_sales` AS

WITH cleaned_and_transformed AS (
  SELECT 
    year,
    
    -- Replace NULL values with clean placeholders
    COALESCE(TRIM(make), 'Unknown') AS make,
    COALESCE(TRIM(model), 'Unknown') AS model,
    COALESCE(TRIM(trim), 'Standard') AS trim,
    COALESCE(TRIM(body), 'Unknown') AS body,
    COALESCE(TRIM(transmission), 'Automatic') AS transmission,
    
    vin,
    state,
    condition,
    odometer,
    color,
    interior,
    COALESCE(TRIM(seller), 'Unknown Seller') AS seller,
    mmr,
    sellingprice,
    saledate AS raw_saledate,

    -- Date parsing without parser errors
    TO_DATE(CONCAT(SUBSTRING(saledate, 5, 6), ' ', SUBSTRING(saledate, 12, 4)), 'MMM dd yyyy') AS sale_date,

    -- Full Day Name
    CASE SUBSTRING(saledate, 1, 3)
      WHEN 'Mon' THEN 'Monday'
      WHEN 'Tue' THEN 'Tuesday'
      WHEN 'Wed' THEN 'Wednesday'
      WHEN 'Thu' THEN 'Thursday'
      WHEN 'Fri' THEN 'Friday'
      WHEN 'Sat' THEN 'Saturday'
      WHEN 'Sun' THEN 'Sunday'
      ELSE 'Unknown'
    END AS day_of_week,

    -- Time Portion
    SUBSTRING(saledate, 17, 8) AS sale_time,

    -- Deduplication by VIN
    ROW_NUMBER() OVER (
      PARTITION BY vin 
      ORDER BY saledate DESC
    ) AS row_num

  FROM `snxb_cardealer`.`brightmotors`.`snxb_car_sales_csv_bright_learn_project`
  WHERE vin IS NOT NULL AND TRIM(vin) != ''
)

SELECT 
  * EXCEPT (row_num)
FROM cleaned_and_transformed
WHERE row_num = 1
ORDER BY sale_date DESC, year DESC; -- Ensures consistent, clean sorting

SELECT * 
FROM `snxb_cardealer`.`brightmotors`.`silver_car_sales` 
LIMIT 1000;

---Filter by Specific Manufacturer or Year  >= 2010
SELECT 
  year, 
  make, 
  model, 
  trim, 
  sellingprice, 
  mmr, 
  sale_date, 
  day_of_week
FROM `snxb_cardealer`.`brightmotors`.`silver_car_sales`
WHERE make = 'Ford' AND year >= 2010
ORDER BY sale_date DESC;

---Filter by Specific Manufacturer or Year <2010
SELECT 
  year, 
  make, 
  model, 
  trim, 
  sellingprice, 
  mmr, 
  sale_date, 
  day_of_week
FROM `snxb_cardealer`.`brightmotors`.`silver_car_sales`
WHERE make = 'Ford' AND year < 2010
ORDER BY sale_date DESC;

---Analyze Sales Patterns by Day of the Week
SELECT 
  day_of_week,
  COUNT(vin) AS total_sales_count,
  ROUND(AVG(sellingprice), 2) AS avg_price
FROM `snxb_cardealer`.`brightmotors`.`silver_car_sales`
GROUP BY day_of_week
ORDER BY total_sales_count DESC;

---Find High-Value or High-Mileage Outliers
SELECT 
  make, 
  model, 
  year, 
  odometer, 
  sellingprice
FROM `snxb_cardealer`.`brightmotors`.`silver_car_sales`
WHERE sellingprice > 50000
ORDER BY sellingprice DESC;

---Most expensive model is Escape
SELECT model, MAX(sellingprice) 
FROM `snxb_cardealer`.`brightmotors`.`silver_car_sales` 
GROUP BY model 
ORDER BY MAX(sellingprice) 
DESC LIMIT 10

---Cheap model is SLS AMG
SELECT model, MIN(sellingprice) 
FROM `snxb_cardealer`.`brightmotors`.`silver_car_sales` 
GROUP BY model 
ORDER BY MIN(sellingprice)
DESC LIMIT 10;

---Total Revenue 7 487 578 138
SELECT SUM(sellingprice) AS total_revenue 
FROM `snxb_cardealer`.`brightmotors`.`silver_car_sales`;

---Total Revenue by Year
SELECT year, SUM(sellingprice) AS total_revenue 
FROM `snxb_cardealer`.`brightmotors`.`silver_car_sales` 
GROUP BY year 
ORDER BY year;

---The top year by Volume (Total Cars Sold) 2012
SELECT 
  year, 
  COUNT(vin) AS total_sales
FROM `snxb_cardealer`.`brightmotors`.`silver_car_sales`
GROUP BY year
ORDER BY total_sales DESC
LIMIT 1;

---The top year by Total Revenue: 2013
SELECT 
  year, 
  SUM(sellingprice) AS total_revenue
FROM `snxb_cardealer`.`brightmotors`.`silver_car_sales`
GROUP BY year
ORDER BY total_revenue DESC
LIMIT 1;


---Most Common Color
SELECT 
DISTINCT(color) AS most_common_color
FROM `snxb_cardealer`.`brightmotors`.`silver_car_sales`
LIMIT 10;

-- Check distinct 'make' entries regardless of case
SELECT 
  LOWER(TRIM(make)) AS normalized_make,
  COLLECT_SET(make) AS variations_found,
  COUNT(*) AS total_records
FROM `snxb_cardealer`.`brightmotors`.`silver_car_sales`
GROUP BY LOWER(TRIM(make))
HAVING SIZE(COLLECT_SET(make)) > 1;


---Fix Casing and Trimming
CREATE OR REPLACE VIEW `snxb_cardealer`.`brightmotors`.`silver_car_sales` AS

WITH cleaned_and_transformed AS (
  SELECT 
    year,
    
    -- Standardize Casing to Proper Case & Strip Spaces
    INITCAP(TRIM(COALESCE(make, 'Unknown'))) AS make,
    INITCAP(TRIM(COALESCE(model, 'Unknown'))) AS model,
    INITCAP(TRIM(COALESCE(trim, 'Standard'))) AS trim,
    INITCAP(TRIM(COALESCE(body, 'Unknown'))) AS body,
    INITCAP(TRIM(COALESCE(transmission, 'Automatic'))) AS transmission,
    INITCAP(TRIM(COALESCE(color, 'Unknown'))) AS color,
    INITCAP(TRIM(COALESCE(interior, 'Unknown'))) AS interior,
    
    vin,
    state,
    condition,
    odometer,
    COALESCE(TRIM(seller), 'Unknown Seller') AS seller,
    mmr,
    sellingprice,
    saledate AS raw_saledate,

    -- Date Parsing
    TO_DATE(CONCAT(SUBSTRING(saledate, 5, 6), ' ', SUBSTRING(saledate, 12, 4)), 'MMM dd yyyy') AS sale_date,

    -- Full Day Name
    CASE SUBSTRING(saledate, 1, 3)
      WHEN 'Mon' THEN 'Monday'
      WHEN 'Tue' THEN 'Tuesday'
      WHEN 'Wed' THEN 'Wednesday'
      WHEN 'Thu' THEN 'Thursday'
      WHEN 'Fri' THEN 'Friday'
      WHEN 'Sat' THEN 'Saturday'
      WHEN 'Sun' THEN 'Sunday'
      ELSE 'Unknown'
    END AS day_of_week,

    -- Time Portion
    SUBSTRING(saledate, 17, 8) AS sale_time,

    -- Deduplication by VIN
    ROW_NUMBER() OVER (
      PARTITION BY vin 
      ORDER BY saledate DESC
    ) AS row_num

  FROM `snxb_cardealer`.`brightmotors`.`snxb_car_sales_csv_bright_learn_project`
  WHERE vin IS NOT NULL AND TRIM(vin) != ''
)

SELECT 
  * EXCEPT (row_num)
FROM cleaned_and_transformed
WHERE row_num = 1
ORDER BY sale_date DESC, year DESC;

SELECT 
  LOWER(TRIM(make)) AS normalized_make,
  COLLECT_SET(make) AS variations_found,
  COUNT(*) AS total_records
FROM `snxb_cardealer`.`brightmotors`.`silver_car_sales`
GROUP BY LOWER(TRIM(make))
HAVING SIZE(COLLECT_SET(make)) > 1;


---Total row count
SELECT 
  (SELECT COUNT(*) FROM `snxb_cardealer`.`brightmotors`.`snxb_car_sales_csv_bright_learn_project`) AS raw_bronze_rows,
  (SELECT COUNT(*) FROM `snxb_cardealer`.`brightmotors`.`silver_car_sales`) AS clean_silver_rows;

 ---For adding columns for analysis
CREATE OR REPLACE VIEW `snxb_cardealer`.`brightmotors`.`silver_car_sales` AS

WITH base_cleaned AS (
  SELECT 
    year,
    INITCAP(TRIM(COALESCE(make, 'Unknown'))) AS make,
    INITCAP(TRIM(COALESCE(model, 'Unknown'))) AS model,
    INITCAP(TRIM(COALESCE(trim, 'Standard'))) AS trim,
    INITCAP(TRIM(COALESCE(body, 'Unknown'))) AS body,
    INITCAP(TRIM(COALESCE(transmission, 'Automatic'))) AS transmission,
    INITCAP(TRIM(COALESCE(color, 'Unknown'))) AS color,
    INITCAP(TRIM(COALESCE(interior, 'Unknown'))) AS interior,
    vin,
    UPPER(TRIM(state)) AS state,
    condition,
    odometer,
    COALESCE(TRIM(seller), 'Unknown Seller') AS seller,
    mmr,
    sellingprice,
    
    -- Date Parsing
    TO_DATE(CONCAT(SUBSTRING(saledate, 5, 6), ' ', SUBSTRING(saledate, 12, 4)), 'MMM dd yyyy') AS sale_date,
    SUBSTRING(saledate, 17, 8) AS sale_time,

    -- Deduplication Sequence
    ROW_NUMBER() OVER (PARTITION BY vin ORDER BY saledate DESC) AS row_num

  FROM `snxb_cardealer`.`brightmotors`.`snxb_car_sales_csv_bright_learn_project`
  WHERE vin IS NOT NULL AND TRIM(vin) != ''
)

SELECT 
  *,
  
  -- Price Calculations
  (sellingprice - mmr) AS price_variance,
  CASE 
    WHEN sellingprice > mmr THEN 'Above Market'
    WHEN sellingprice < mmr THEN 'Below Market'
    ELSE 'At Market'
  END AS deal_quality,
  CASE 
    WHEN sellingprice < 10000 THEN 'Budget (<$10k)'
    WHEN sellingprice BETWEEN 10000 AND 25000 THEN 'Mid-Range ($10k-$25k)'
    WHEN sellingprice BETWEEN 25001 AND 50000 THEN 'Premium ($25k-$50k)'
    ELSE 'Luxury (>$50k)'
  END AS price_tier,

  -- Vehicle Metrics
  (YEAR(sale_date) - year) AS vehicle_age,
  CASE 
    WHEN odometer < 30000 THEN 'Low Mileage (<30k)'
    WHEN odometer BETWEEN 30000 AND 80000 THEN 'Moderate Mileage (30k-80k)'
    WHEN odometer BETWEEN 80001 AND 150000 THEN 'High Mileage (80k-150k)'
    ELSE 'Very High Mileage (>150k)'
  END AS mileage_tier,

  -- Date Features for Slicers
  DATE_FORMAT(sale_date, 'yyyy-MM') AS sale_year_month,
  CONCAT('Q', QUARTER(sale_date)) AS sale_quarter,
  CASE WHEN DAYOFWEEK(sale_date) IN (1, 7) THEN 'Weekend' ELSE 'Weekday' END AS day_type

FROM base_cleaned
WHERE row_num = 1;

---Viewing added columns
SELECT 
  make,
  model,
  sellingprice,
  mmr,
  price_variance,
  deal_quality,
  price_tier,
  vehicle_age,
  mileage_tier,
  sale_year_month,
  sale_quarter,
  day_type
FROM `snxb_cardealer`.`brightmotors`.`silver_car_sales`
LIMIT 5;

---Viewing the final table
SELECT * 
FROM `snxb_cardealer`.`brightmotors`.`silver_car_sales`; 
